# SI Notebook 1: Shared OrbitalAnalyzer payload

This supplementary notebook shows the common orbital-analysis layer used by the article. The example is ethylene because its sigma and pi candidates make the bridge between NAO/NPA analysis, NBO assignment, and VB active-space construction visible in a compact calculation.

In [ ]:
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
import importlib.util
import io
import sys

import numpy as np

from veloxchem.molecule import Molecule
from veloxchem.molecularbasis import MolecularBasis
from veloxchem.scfrestdriver import ScfRestrictedDriver

start = Path.cwd().resolve()
repo_root = next(path for path in (start, *start.parents) if (path / 'src' / 'pymodule').exists())
for module_name in ('orbitalanalyzerdriver', 'nbodriver', 'vbdriver'):
    module_path = repo_root / 'src' / 'pymodule' / f'{module_name}.py'
    spec = importlib.util.spec_from_file_location(f'veloxchem.{module_name}', module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f'veloxchem.{module_name}'] = module
    spec.loader.exec_module(module)

from veloxchem.orbitalanalyzerdriver import OrbitalAnalyzer, OrbitalAnalyzerOptions
from veloxchem.nbodriver import NboComputeOptions, NboDriver
from veloxchem.vbdriver import VbComputeOptions, VbDriver

@contextmanager
def quiet_veloxchem():
    with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
        yield

def run_rhf(molecule, basis_label='sto-3g'):
    basis = MolecularBasis.read(molecule, basis_label, ostream=None)
    scf = ScfRestrictedDriver()
    scf.ostream.mute()
    scf.xcfun = 'hf'
    with quiet_veloxchem():
        scf.compute(molecule, basis)
    return basis, scf

driver_source_dir = repo_root / 'src' / 'pymodule'
print(f'Using local driver sources from {driver_source_dir}')


## Ethylene reference calculation

The OrbitalAnalyzer is run after a small RHF calculation. The result object stores AO maps, the density and overlap matrices, NAO/NPA data, optional MO decomposition, and the shared orbital-candidate list.

In [ ]:
ethylene = Molecule.read_str('''
C -0.66950000  0.00000000 0.00000000
C  0.66950000  0.00000000 0.00000000
H -1.23210000  0.92890000 0.00000000
H -1.23210000 -0.92890000 0.00000000
H  1.23210000  0.92890000 0.00000000
H  1.23210000 -0.92890000 0.00000000
''')
ethylene.set_charge(0)
ethylene.set_multiplicity(1)

basis, scf = run_rhf(ethylene, basis_label='sto-3g')
analysis = OrbitalAnalyzer(
    ethylene,
    basis,
    mol_orbs=scf.mol_orbs,
    options=OrbitalAnalyzerOptions(mo_analysis_top=4, mo_analysis_threshold=0.03),
).run()

summary = {
    'nao_count': int(len(analysis.nao_data.populations)),
    'candidate_count': len(analysis.orbital_candidates),
    'equivalent_atom_groups': analysis.equivalent_atom_groups,
    'has_spin_data': analysis.spin_data is not None,
}
summary

## NAO decomposition and candidate table

The first table is the NAO population decomposition used downstream by the NBO and VB layers. The second table shows the highest-occupation shared candidates. Candidate atom indices in the driver payload are zero based; the printed table shifts them to one-based labels for the article and supplement.

In [ ]:
labels = ethylene.get_labels()
l_names = {0: 's', 1: 'p', 2: 'd', 3: 'f'}
nao_rows = [
    {
        'nao': index + 1,
        'atom': f'{labels[int(atom)]}{int(atom) + 1}',
        'l': l_names.get(int(l_value), str(int(l_value))),
        'population': round(float(pop), 6),
    }
    for index, (atom, l_value, pop) in enumerate(
        zip(analysis.nao_data.atom_map, analysis.nao_data.angular_momentum_map, analysis.nao_data.populations)
    )
]

candidate_rows = [
    {
        'index': candidate.get('index'),
        'type': candidate.get('type'),
        'subtype': candidate.get('subtype'),
        'atoms': tuple(atom + 1 for atom in candidate.get('atoms', ())),
        'occupation': round(float(candidate.get('occupation', 0.0)), 6),
        'source': candidate.get('source'),
    }
    for candidate in sorted(
        analysis.orbital_candidates,
        key=lambda item: item.get('occupation', 0.0),
        reverse=True,
    )[:12]
]

nao_rows[:12], candidate_rows

## Same payload feeding NBO and VB

The NBO driver internally rebuilds the same OrbitalAnalyzer payload and then applies Lewis assignment and resonance diagnostics. The VB driver can also use the same candidate vocabulary to define the active pi bond. This is the API connection described in the implementation section of the article.

In [ ]:
nbo = NboDriver()
nbo.verbose = False
nbo_results = nbo.compute(
    ethylene,
    basis,
    scf.mol_orbs,
    mode='primary',
    options=NboComputeOptions(max_alternatives=4, mo_analysis_top=4),
)

vb_results = VbDriver().compute(
    ethylene,
    basis,
    reference_orbitals=scf.mol_orbs,
    options=VbComputeOptions(
        mode='vbscf',
        active_bond=(0, 1),
        active_candidate_subtype='pi',
        include_ionic=True,
        freeze_inactive_orbitals=True,
        orbital_relaxation_symmetry='equivalent-centers',
        orbital_amplitude_bound=0.05,
    ),
)

bridge_summary = {
    'nbo_candidate_count': len(nbo_results['nbo_candidates']),
    'nbo_primary_counts': nbo_results['primary']['counts'],
    'vb_energy_hartree': float(vb_results['energy']),
    'vb_active_candidate': vb_results.get('active_space', {}).get('active_candidate_label'),
    'vb_weights': vb_results.get('weights'),
}
bridge_summary